# AABB Calculation

In [1]:
import torch
import numpy as np
import plotly.graph_objects as go
from geocutool.primitive.gs import compute_gaussian_aabb

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

means = torch.tensor([[0.5, 0.5, 0.5]]).to(device)
rotations = torch.tensor([[0.5, 0.2, 0.7, 0.3]]).to(device)
scales = torch.tensor([[0.1, 0.6, 0.5]]).to(device)
opacities = torch.tensor([0.5]).to(device)

level = 8
iso = 11.345
tol = 1. / 8.
rotnorm = True

aabb_min, aabb_max, contact_points, covis = compute_gaussian_aabb(
    means, 
    rotations, 
    scales, 
    iso=iso, 
    tol=tol, 
    level=level, 
    rotnorm=rotnorm
)

print("AABB Min:", aabb_min.shape)
print("AABB Max:", aabb_max.shape)
print("Contact Points:", contact_points.shape)
print("Covariance Inverses:", covis.shape)

mean_np = means[0].cpu().numpy()
rot_np = rotations[0].cpu().numpy()
scale_np = scales[0].cpu().numpy()

min_pt = aabb_min[0].cpu().numpy()
max_pt = aabb_max[0].cpu().numpy()

contact_points_np = contact_points[0].cpu().numpy()
covis_np = covis[0].cpu().numpy()

all_contact_points = np.concatenate([contact_points_np, -contact_points_np], axis=0)

voxelSize = 2.0 / (1 << level)
min_scale = tol * voxelSize
scale_np = np.maximum(scale_np, min_scale)

u = np.linspace(0, 2 * np.pi, 50)
v = np.linspace(0, np.pi, 50)
x_sphere = np.outer(np.cos(u), np.sin(v))
y_sphere = np.outer(np.sin(u), np.sin(v))
z_sphere = np.outer(np.ones(np.size(u)), np.cos(v))

rx, ry, rz = scale_np * np.sqrt(iso)
x_ellipsoid = x_sphere * rx
y_ellipsoid = y_sphere * ry
z_ellipsoid = z_sphere * rz

points = np.vstack(
    [
        x_ellipsoid.flatten(), 
        y_ellipsoid.flatten(), 
        z_ellipsoid.flatten()
    ]
)

r, x, y, z = rot_np[0], rot_np[1], rot_np[2], rot_np[3]
if rotnorm:
    inv_norm = 1.0 / np.sqrt(r*r + x*x + y*y + z*z)
    r *= inv_norm; x *= inv_norm; y *= inv_norm; z *= inv_norm

R_mat = np.array([
    [1 - 2*(y*y + z*z), 2*(x*y - r*z), 2*(x*z + r*y)],
    [2*(x*y + r*z), 1 - 2*(x*x + z*z), 2*(y*z - r*x)],
    [2*(x*z - r*y), 2*(y*z + r*x), 1 - 2*(x*x + y*y)]
])

rotated_points = R_mat @ points
x_final = rotated_points[0, :].reshape(x_sphere.shape) + mean_np[0]
y_final = rotated_points[1, :].reshape(y_sphere.shape) + mean_np[1]
z_final = rotated_points[2, :].reshape(z_sphere.shape) + mean_np[2]

xc = [
    min_pt[0], max_pt[0], max_pt[0], min_pt[0], 
    min_pt[0], max_pt[0], max_pt[0], min_pt[0]
]
yc = [
    min_pt[1], min_pt[1], max_pt[1], max_pt[1], 
    min_pt[1], min_pt[1], max_pt[1], max_pt[1]
]
zc = [
    min_pt[2], min_pt[2], min_pt[2], min_pt[2], 
    max_pt[2], max_pt[2], max_pt[2], max_pt[2]
]

x_lines = [
    xc[0], xc[1], xc[2], xc[3], xc[0], None, 
    xc[4], xc[5], xc[6], xc[7], xc[4], None, 
    xc[0], xc[4], None, xc[1], xc[5], None, 
    xc[2], xc[6], None, xc[3], xc[7]]
y_lines = [
    yc[0], yc[1], yc[2], yc[3], yc[0], None, 
    yc[4], yc[5], yc[6], yc[7], yc[4], None, 
    yc[0], yc[4], None, yc[1], yc[5], None, 
    yc[2], yc[6], None, yc[3], yc[7]]
z_lines = [
    zc[0], zc[1], zc[2], zc[3], zc[0], None, 
    zc[4], zc[5], zc[6], zc[7], zc[4], None, 
    zc[0], zc[4], None, zc[1], zc[5], None,
    zc[2], zc[6], None, zc[3], zc[7]]

fig = go.Figure()

fig.add_trace(go.Surface(
    x=x_final, y=y_final, z=z_final, 
    opacity=0.6, 
    colorscale='Plasma', 
    showscale=False,
    name='Gaussian'
))

fig.add_trace(go.Scatter3d(
    x=x_lines, y=y_lines, z=z_lines,
    mode='lines',
    line=dict(color='red', width=4),
    name='AABB'
))

fig.add_trace(go.Scatter3d(
    x=all_contact_points[0::3] + mean_np[0],
    y=all_contact_points[1::3] + mean_np[1],
    z=all_contact_points[2::3] + mean_np[2],
    mode='markers',
    marker=dict(size=5, color='green'),
    name='Contact Points'
))

fig.update_layout(
    scene=dict(aspectmode='data'),
    scene_camera=dict(projection=dict(type="orthographic")),
    title="3D Gaussian and Computed AABB",
    # margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()

AABB Min: torch.Size([1, 3])
AABB Max: torch.Size([1, 3])
Contact Points: torch.Size([1, 9])
Covariance Inverses: torch.Size([1, 6])


# Gaussian Voxel Intersection (Brute Force)

In [2]:

from geocutool.primitive.gs import query_gs_voxel_intersection

def generate_nxnxn_voxel_grid(n=3, voxel_size=0.1, center=(0.0, 0.0, 0.0), device='cuda'):
    """
    Generates an n x n x n grid of voxel AABBs centered at a specific coordinate.
    Returns vx_aabb_mins, vx_aabb_maxs of shape (n**3, 3).
    """
    # Create dynamic steps to keep the grid perfectly centered.
    # For n=3 -> [-1.0,  0.0,  1.0]
    # For n=4 -> [-1.5, -0.5,  0.5,  1.5]
    steps = torch.arange(n, dtype=torch.float32, device=device) - (n - 1) / 2.0
    
    # Create a 3D meshgrid
    Z, Y, X = torch.meshgrid(steps, steps, steps, indexing='ij')
    
    # Flatten the grid and scale by voxel size to get the center coordinates of each voxel
    centers = torch.stack(
        [
            X.flatten(), 
            Y.flatten(), 
            Z.flatten()
        ], dim=-1
    ) * voxel_size
    
    # Shift the entire grid to the requested center
    center_tensor = torch.tensor(center, dtype=torch.float32, device=device)
    centers += center_tensor
    
    # Calculate AABB min and max by subtracting/adding half the voxel size
    half_size = voxel_size / 2.0
    vx_aabb_mins = centers - half_size
    vx_aabb_maxs = centers + half_size
    
    return vx_aabb_mins.contiguous(), vx_aabb_maxs.contiguous()

def generate_batch_wireframe(mins, maxs):
    """
    Takes N voxel min/max tensors, returns single flattened lists of x, y, z coords
    with None separators to draw all boxes in a single Plotly trace.
    """
    x_lines, y_lines, z_lines = [], [], []
    
    # Convert to numpy if they are tensors
    mins_np = mins.cpu().numpy()
    maxs_np = maxs.cpu().numpy()
    
    for min_pt, max_pt in zip(mins_np, maxs_np):
        # 8 corners of the voxel
        xc = [
            min_pt[0], max_pt[0], max_pt[0], min_pt[0], 
            min_pt[0], max_pt[0], max_pt[0], min_pt[0]]
        yc = [
            min_pt[1], min_pt[1], max_pt[1], max_pt[1], 
            min_pt[1], min_pt[1], max_pt[1], max_pt[1]]
        zc = [
            min_pt[2], min_pt[2], min_pt[2], min_pt[2], 
            max_pt[2], max_pt[2], max_pt[2], max_pt[2]]

        # The 12 edge drawing path (with Nones to lift the pen)
        x_lines.extend([
            xc[0], xc[1], xc[2], xc[3], xc[0], None, 
            xc[4], xc[5], xc[6], xc[7], xc[4], None, 
            xc[0], xc[4], None, xc[1], xc[5], None, 
            xc[2], xc[6], None, xc[3], xc[7], None])
        y_lines.extend([
            yc[0], yc[1], yc[2], yc[3], yc[0], None, 
            yc[4], yc[5], yc[6], yc[7], yc[4], None, 
            yc[0], yc[4], None, yc[1], yc[5], None, 
            yc[2], yc[6], None, yc[3], yc[7], None])
        z_lines.extend([
            zc[0], zc[1], zc[2], zc[3], zc[0], None, 
            zc[4], zc[5], zc[6], zc[7], zc[4], None, 
            zc[0], zc[4], None, zc[1], zc[5], None, 
            zc[2], zc[6], None, zc[3], zc[7], None])
        
    return x_lines, y_lines, z_lines

vx_aabb_mins, vx_aabb_maxs = generate_nxnxn_voxel_grid(
    n=8,
    voxel_size=0.5, 
    center=(0.0, 0.0, 0.0), 
    device=device
)

ar_threshold = 0.1
p_threshold = 0.1
return_centroids = True

print("Voxel AABB Mins:\n", vx_aabb_mins.shape)
print("Voxel AABB Maxs:\n", vx_aabb_maxs.shape)

hit_mask, out_voxel_ids, out_gaus_ids, centroids, densities = query_gs_voxel_intersection(
    vx_aabb_mins,
    vx_aabb_maxs,
    means,
    covis,
    opacities,
    aabb_min, 
    aabb_max, 
    contact_points,
    iso,
    ar_threshold,
    p_threshold,
    return_centroids
)

print("Hit Mask:", hit_mask.shape)
print("Output Voxel IDs:", out_voxel_ids.shape)
print("Output Gaussian IDs:", out_gaus_ids.shape)
print("Centroids:", centroids.shape if return_centroids else "Not Returned")
print("Densities:", densities.shape if return_centroids else "Not Returned")
print(f"Num Hits: {hit_mask.sum().item()}")
for i in range(out_voxel_ids.shape[0]):
    print(f"Voxel ID: {out_voxel_ids[i].item()}, Gaussian ID: {out_gaus_ids[i].item()}, Centroid: {centroids[i].cpu().numpy() if return_centroids else 'N/A'}, Density: {densities[i].item() if return_centroids else 'N/A'}")

# Contruct voxel wireframe for visualization

ivx_aabb_mins = vx_aabb_mins[hit_mask]
ivx_aabb_maxs = vx_aabb_maxs[hit_mask]

nivx_aabb_mins = vx_aabb_mins[~hit_mask]
nivx_aabb_maxs = vx_aabb_maxs[~hit_mask]

hit_x, hit_y, hit_z = generate_batch_wireframe(ivx_aabb_mins, ivx_aabb_maxs)
miss_x, miss_y, miss_z = generate_batch_wireframe(nivx_aabb_mins, nivx_aabb_maxs)

if len(miss_x) > 0:
    fig.add_trace(go.Scatter3d(
        x=miss_x, y=miss_y, z=miss_z,
        mode='lines',
        line=dict(color='rgba(0, 0, 255, 0.3)', width=2), # Lower opacity blue
        name='Missed Voxels'
    ))

# Add Hit Voxels (Green)
if len(hit_x) > 0:
    fig.add_trace(go.Scatter3d(
        x=hit_x, y=hit_y, z=hit_z,
        mode='lines',
        line=dict(color='rgba(0, 255, 0, 0.8)', width=4), # Bright green
        name='Hit Voxels'
    ))

# Update layout for a better view of the grid
fig.update_layout(
    title="Gaussian vs. Voxel Grid Intersection",
    width=800,
    height=800,
    scene=dict(
        aspectmode='data',
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    )
)

fig.show()

Voxel AABB Mins:
 torch.Size([512, 3])
Voxel AABB Maxs:
 torch.Size([512, 3])
Hit Mask: torch.Size([512])
Output Voxel IDs: torch.Size([111])
Output Gaussian IDs: torch.Size([111])
Centroids: torch.Size([111, 3])
Densities: torch.Size([111])
Num Hits: 111
Voxel ID: 149, Gaussian ID: 0, Centroid: [ 0.75       -0.74826854 -0.75      ], Density: 0.004231208469718695
Voxel ID: 157, Gaussian ID: 0, Centroid: [ 0.75 -0.25 -0.75], Density: 0.001079017180018127
Voxel ID: 212, Gaussian ID: 0, Centroid: [ 0.25       -0.74826854 -0.25      ], Density: 0.001083784969523549
Voxel ID: 220, Gaussian ID: 0, Centroid: [ 0.25 -0.25 -0.25], Density: 0.06575526297092438
Voxel ID: 221, Gaussian ID: 0, Centroid: [ 0.75 -0.25 -0.25], Density: 0.0667421892285347
Voxel ID: 229, Gaussian ID: 0, Centroid: [ 0.75  0.25 -0.25], Density: 0.008606134913861752
Voxel ID: 428, Gaussian ID: 0, Centroid: [0.25 0.75 1.25], Density: 0.008606134913861752
Voxel ID: 436, Gaussian ID: 0, Centroid: [0.25 1.25 1.25], Density: 0.

In [3]:
for name in ["Missed Voxels", "Contact Points", "AABB"]:
    fig.update_traces(visible=False, selector=dict(name=name))

hit_vx_mins = vx_aabb_mins[out_voxel_ids]
hit_vx_maxs = vx_aabb_maxs[out_voxel_ids]
hit_gs_mins = aabb_min[out_gaus_ids]
hit_gs_maxs = aabb_max[out_gaus_ids]

overlap_mins = torch.max(hit_vx_mins, hit_gs_mins)
overlap_maxs = torch.min(hit_vx_maxs, hit_gs_maxs)

overlap_x, overlap_y, overlap_z = generate_batch_wireframe(overlap_mins, overlap_maxs)

if len(overlap_x) > 0:
    fig.add_trace(go.Scatter3d(
        x=overlap_x, y=overlap_y, z=overlap_z,
        mode='lines',
        line=dict(color='rgba(255, 165, 0, 1.0)', width=5), # Thick Orange
        name='Overlap Boxes'
    ))

if return_centroids and centroids is not None and len(centroids) > 0:
    centroids_np = centroids.cpu().numpy()
    densities_np = densities.cpu().numpy()
    
    fig.add_trace(go.Scatter3d(
        x=centroids_np[:, 0],
        y=centroids_np[:, 1],
        z=centroids_np[:, 2],
        mode='markers',
        marker=dict(
            color=densities_np,       # Map the color to the exact density value!
            colorscale='Plasma',      # 'Plasma' goes from very dark purple (0.0) to bright yellow (high)
            cmin=0.0,                 # Force 0.0 to be the darkest color
            cmax=float(densities_np.max()), # Force the max density to be the brightest color
            showscale=True,           # Adds a colorbar legend to the right side of the screen
            colorbar=dict(title="Density", thickness=15, len=0.5, xpad=50),
            size=6,
            symbol='diamond',
            line=dict(color='rgba(255, 255, 255, 0.5)', width=1) # Slight white outline
        ),
        name='Approximated Centroids'
    ))

fig.show()

In [4]:
for name in ["Approximated Centroids", "Hit Voxels"]:
    fig.update_traces(visible=False, selector=dict(name=name))

if return_centroids and centroids is not None and len(centroids) > 0:
    centroids_np = centroids.cpu().numpy()
    densities_np = densities.cpu().numpy()
    
    # Create the filter mask
    valid_mask = densities_np > 0.001
    
    # Apply the mask to BOTH arrays to keep the coordinates and colors perfectly aligned
    meaningful_centroids = centroids_np[valid_mask]
    meaningful_densities = densities_np[valid_mask]
    
    # Ensure there are still points left to plot after filtering!
    if len(meaningful_centroids) > 0:
        fig.add_trace(go.Scatter3d(
            x=meaningful_centroids[:, 0],
            y=meaningful_centroids[:, 1],
            z=meaningful_centroids[:, 2],
            mode='markers',
            marker=dict(
                color=meaningful_densities,       
                colorscale='Plasma',      
                # Keeping cmin at 0.0 keeps the color scale visually consistent with your previous plots
                cmin=0.0,                 
                cmax=float(meaningful_densities.max()), 
                showscale=True,           
                colorbar=dict(title="Density", thickness=15, len=0.5, xpad=50),
                size=6,
                symbol='diamond',
                line=dict(color='rgba(255, 255, 255, 0.5)', width=1) 
            ),
            name='Approximated Centroids (>0.001)'
        ))

fig.show()

In [5]:
if return_centroids and densities is not None and len(densities) > 0:
    densities_np = densities.cpu().numpy()

    # 2. Create an interactive Histogram
    fig_hist = go.Figure(data=[go.Histogram(
        x=densities_np,
        nbinsx=100, # Bins the data into 100 vertical bars
        marker_color='rgba(255, 165, 0, 0.8)', # Orange to match your overlap boxes
        opacity=0.75
    )])

    # 3. Update the layout for clarity
    fig_hist.update_layout(
        title_text='Distribution of Gaussian Intersection Densities', # title of plot
        xaxis_title_text='Calculated Density (Alpha)', # xaxis label
        yaxis_title_text='Frequency (Number of Intersections)', # yaxis label
        bargap=0.1, # Gap between bars so it looks clean
        template='plotly_dark' # Optional: makes it look like a sleek terminal UI
    )

    fig_hist.show()
else:
    print("Densities tensor is empty or return_centroids was set to False.")

# Gaussian Edge Intersection (Brute Force)

In [6]:
from geocutool.primitive.gs import query_gs_edge_intersection
from typing import Tuple, Optional

for name in ["Approximated Centroids (>0.001)", "Overlap Boxes"]:
    fig.update_traces(visible=False, selector=dict(name=name))

def generate_unique_aabb_edges(
    mins: torch.Tensor, 
    maxs: torch.Tensor, 
    return_mapping: bool = True
) -> Tuple[torch.Tensor, torch.Tensor, Optional[torch.Tensor]]:
    """
    Generates the unique wireframe edges for a batch of AABBs, 
    stripping out any overlapping edges shared by adjacent voxels.
    """
    # 1. Extract X, Y, Z components
    x0, y0, z0 = mins[:, 0:1], mins[:, 1:2], mins[:, 2:3]
    x1, y1, z1 = maxs[:, 0:1], maxs[:, 1:2], maxs[:, 2:3]
    
    # 2. Define the 8 corners for all M boxes
    c000 = torch.cat([x0, y0, z0], dim=-1).unsqueeze(1)
    c100 = torch.cat([x1, y0, z0], dim=-1).unsqueeze(1)
    c010 = torch.cat([x0, y1, z0], dim=-1).unsqueeze(1)
    c110 = torch.cat([x1, y1, z0], dim=-1).unsqueeze(1)
    c001 = torch.cat([x0, y0, z1], dim=-1).unsqueeze(1)
    c101 = torch.cat([x1, y0, z1], dim=-1).unsqueeze(1)
    c011 = torch.cat([x0, y1, z1], dim=-1).unsqueeze(1)
    c111 = torch.cat([x1, y1, z1], dim=-1).unsqueeze(1)
    
    # 3. Build the Edges
    sx = torch.cat([c000, c010, c001, c011], dim=1)
    ex = torch.cat([c100, c110, c101, c111], dim=1)
    sy = torch.cat([c000, c100, c001, c101], dim=1)
    ey = torch.cat([c010, c110, c011, c111], dim=1)
    sz = torch.cat([c000, c100, c010, c110], dim=1)
    ez = torch.cat([c001, c101, c011, c111], dim=1)
    
    # 4. Concatenate and flatten
    edge_starts_flat = torch.cat([sx, sy, sz], dim=1).view(-1, 3)
    edge_ends_flat = torch.cat([ex, ey, ez], dim=1).view(-1, 3)
    
    # 5. Create a 6D Signature for each edge (start_x, start_y, start_z, end_x, end_y, end_z)
    edge_signatures = torch.cat([edge_starts_flat, edge_ends_flat], dim=-1)
    
    # 6. Filter for Uniqueness!
    if return_mapping:
        # inverse_indices maps our compressed unique list back to the original naive list
        unique_edges, inverse_indices = torch.unique(edge_signatures, dim=0, return_inverse=True)
        unique_starts = unique_edges[:, 0:3].contiguous()
        unique_ends = unique_edges[:, 3:6].contiguous()
        return unique_starts, unique_ends, inverse_indices
    else:
        unique_edges = torch.unique(edge_signatures, dim=0)
        unique_starts = unique_edges[:, 0:3].contiguous()
        unique_ends = unique_edges[:, 3:6].contiguous()
        return unique_starts, unique_ends, None
    
def build_plotly_lines(starts, ends):
    starts_np = starts.cpu().numpy()
    ends_np = ends.cpu().numpy()
    
    x_lines, y_lines, z_lines = [], [], []
    for s, e in zip(starts_np, ends_np):
        x_lines.extend([s[0], e[0], None])
        y_lines.extend([s[1], e[1], None])
        z_lines.extend([s[2], e[2], None])
    return x_lines, y_lines, z_lines

edge_starts, edge_ends, inverse_indices = generate_unique_aabb_edges(
    vx_aabb_mins, vx_aabb_maxs, return_mapping=True
)

print("Edge Starts Shape:", edge_starts.shape)
print("Edge Ends Shape:", edge_ends.shape)

hit_mask, out_edge_ids, out_gaus_ids = query_gs_edge_intersection(
    edge_starts,
    edge_ends,
    means,
    covis,
    iso=iso
)

print("Hit Mask:", hit_mask.shape)
print("Output Edge IDs:", out_edge_ids.shape)
print("Output Gaussian IDs:", out_gaus_ids.shape)
print(f"Num Edge Hits: {hit_mask.sum().item()}")

hit_edge_indices = torch.unique(out_edge_ids)

num_total_edges = edge_starts.shape[0]
is_hit_mask = torch.zeros(num_total_edges, dtype=torch.bool, device=edge_starts.device)

if hit_edge_indices.numel() > 0:
    is_hit_mask[hit_edge_indices] = True
    
hit_starts = edge_starts[is_hit_mask]
hit_ends = edge_ends[is_hit_mask]

miss_starts = edge_starts[~is_hit_mask]
miss_ends = edge_ends[~is_hit_mask]

hit_x, hit_y, hit_z = build_plotly_lines(hit_starts, hit_ends)
miss_x, miss_y, miss_z = build_plotly_lines(miss_starts, miss_ends)

# if len(miss_x) > 0:
#     fig.add_trace(go.Scatter3d(
#         x=miss_x, y=miss_y, z=miss_z,
#         mode='lines',
#         line=dict(color='rgba(0, 0, 255, 0.2)', width=2), # Low opacity blue so it doesn't clutter
#         name='Missed Edges'
#     ))

# Add Hit Edges (Green)
if len(hit_x) > 0:
    fig.add_trace(go.Scatter3d(
        x=hit_x, y=hit_y, z=hit_z,
        mode='lines',
        line=dict(color='rgba(0, 255, 0, 1.0)', width=5), # Bright, thick green
        name='Hit Edges'
    ))

# Keep the layout clean
fig.update_layout(
    title="Gaussian Wireframe Edge Intersections",
    width=800,
    height=800,
    scene=dict(
        aspectmode='data',
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        # bgcolor='rgb(20, 20, 20)' # Dark background makes the glowing lines pop
    ),
    # template='plotly_dark'
)

fig.show()

Edge Starts Shape: torch.Size([1944, 3])
Edge Ends Shape: torch.Size([1944, 3])
Hit Mask: torch.Size([1944])
Output Edge IDs: torch.Size([162])
Output Gaussian IDs: torch.Size([162])
Num Edge Hits: 162
